# Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

# Create a working folder on your Drive
import os
os.makedirs('/content/drive/MyDrive/DPCC', exist_ok=True)

Mounted at /content/drive


# Install Miniconda into Colab (to get Python 3.10)

In [2]:
%%bash
# Install Miniconda (gives us conda + Python 3.10 control)
if [ ! -f "/content/miniconda.sh" ]; then
    wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /content/miniconda.sh
fi
bash /content/miniconda.sh -b -p /content/miniconda3 -u
echo "✓ Miniconda installed"
/content/miniconda3/bin/conda --version

PREFIX=/content/miniconda3
Unpacking bootstrapper...
Unpacking payload...

Installing base environment...

Preparing transaction: ...working... done
Executing transaction: ...working... done
installation finished.
    You currently have a PYTHONPATH environment variable set. This may cause
    unexpected behavior when running the Python interpreter in Miniconda3.
    For best results, please verify that your PYTHONPATH only points to
    directories of packages that are compatible with the Python interpreter
    in Miniconda3: /content/miniconda3
✓ Miniconda installed
conda 26.1.1


# Clone the DPCC Repo (to Drive for persistence)

In [3]:
%%bash
cd /content/drive/MyDrive/DPCC # Corrected directory name

# Only clone if not already there
if [ ! -d "dpcc" ]; then
    git clone --recurse-submodules "https://github.com/ghubliming/dpcc.git" # Added quotes to the URL
    echo "Cloned successfully."
else
    echo "Repo already exists, pulling latest..."
    cd dpcc && git pull
fi

Cloned successfully.


Cloning into 'dpcc'...
Updating files: 100% (199/199), done.


# Create conda env with Python 3.10

In [7]:
%%bash
# Accept Conda Terms of Service for required channels
/content/miniconda3/bin/conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
/content/miniconda3/bin/conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r

# Create env exactly as README says
/content/miniconda3/bin/conda create -n dpcc python=3.10 -y -q
echo "✓ conda env 'dpcc' created with Python 3.10"
/content/miniconda3/envs/dpcc/bin/python --version

accepted Terms of Service for https://repo.anaconda.com/pkgs/main
accepted Terms of Service for https://repo.anaconda.com/pkgs/r
Jupyter detected...
2 channel Terms of Service accepted
Retrieving notices: ...working... done
Channels:
 - defaults
Platform: linux-64
Solving environment: ...working... done

## Package Plan ##

  environment location: /content/miniconda3/envs/dpcc

  added / updated specs:
    - python=3.10


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    ld_impl_linux-64-2.44      |       h9e0c5a2_3         725 KB
    libnsl-2.0.0               |       h5eee18b_0          31 KB
    packaging-25.0             |  py310h06a4308_1         164 KB
    python-3.10.20             |       h741d88c_0        24.1 MB
    setuptools-80.10.2         |  py310h06a4308_0         1.3 MB
    sqlite-3.51.2              |       h3e8d24a_0         1.2 MB
    tzdata-2026a               |       h

# Clone D3IL

In [8]:
%%bash
# 1. Ensure the directory exists before jumping in
mkdir -p /content/drive/MyDrive/DPCC/dpcc/

D3IL="/content/drive/MyDrive/DPCC/dpcc/d3il"
# Double check this path exists!
PIP="/content/miniconda3/envs/dpcc/bin/pip"

if [ ! -f "$PIP" ]; then
    echo "✗ Critical Error: Pip not found at $PIP. Check your Miniconda installation."
    exit 1
fi

# 2. Use a more robust check for the git repo
if [ ! -d "$D3IL/.git" ]; then
    echo "d3il missing or incomplete — cleaning and cloning..."
    rm -rf "$D3IL"
    git clone https://github.com/ALRhub/d3il.git "$D3IL"
else
    echo "✓ d3il repository detected."
fi

# 3. Use absolute paths for the pip install
$PIP install "$D3IL/environments/d3il"
$PIP install "$D3IL/environments/d3il/envs/gym_avoiding_env"

d3il missing or incomplete — cleaning and cloning...
Processing ./drive/MyDrive/DPCC/dpcc/d3il/environments/d3il
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for environments.d3il.d3il_sim: filename=environments_d3il_d3il_sim-0.2-py3-none-any.whl size=159418 sha256=fbc43bb41f42878686e374552041c37adcfd44dd7c66416cd4f666c9a48f32b5
  Stored in directory: /tmp/pip-ephem-wheel-cache-zwpvjkiw/wheels/19/38/11/f339016918b730140ec866d0486669dd65174da86a5eae116e
Successfully built environments.d3il.d3il_sim
Processing ./drive/MyDrive/DPCC/dpcc/d3il/environments/d3il/envs/gym_avoiding_env
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting re

Cloning into '/content/drive/MyDrive/DPCC/dpcc/d3il'...
Updating files: 100% (896/896), done.


# Install PyTorch (CUDA124)

In [9]:
%%bash
/content/miniconda3/envs/dpcc/bin/pip install \
    torch torchvision \
    --index-url https://download.pytorch.org/whl/cu124 -q
echo "✓ PyTorch installed"
/content/miniconda3/envs/dpcc/bin/python -c \
    "import torch; print('PyTorch:', torch.__version__); print('CUDA:', torch.cuda.is_available())"

✓ PyTorch installed
PyTorch: 2.6.0+cu124
CUDA: True


# Install All Requirements

In [10]:
%%bash
PIP="/content/miniconda3/envs/dpcc/bin/pip"
cd /content/drive/MyDrive/DPCC/dpcc

echo "========================================="
echo " INSTALLING FROM requirements.txt"
echo "========================================="
echo "Using pip: $($PIP --version)"
echo ""

if [ ! -f requirements.txt ]; then
  echo "ERROR: requirements.txt not found in $(pwd)"
  exit 1
fi

echo "Found requirements.txt -- $(wc -l < requirements.txt) packages listed"
echo ""

PASS=0
FAIL=0
SKIP=0
FAILED_PKGS=()

while IFS= read -r line || [ -n "$line" ]; do
  [[ -z "$line" || "$line" == \#* ]] && continue

  PKG="$line"
  printf "  %-40s" "$PKG"

  RESULT=$($PIP install "$PKG" -q 2>&1)
  STATUS=$?

  if [ $STATUS -eq 0 ]; then
    if echo "$RESULT" | grep -q "already satisfied"; then
      echo "[ already installed ]"
      ((SKIP++))
    else
      echo "[ installed ]"
      ((PASS++))
    fi
  else
    echo "[ FAILED ]"
    echo "    └─ $(echo "$RESULT" | tail -3)"
    FAILED_PKGS+=("$PKG")
    ((FAIL++))
  fi

done < requirements.txt

echo ""
echo "========================================="
echo " SUMMARY"
echo "========================================="
printf "  %-25s %d\n" "Newly installed:"  "$PASS"
printf "  %-25s %d\n" "Already installed:" "$SKIP"
printf "  %-25s %d\n" "Failed:"           "$FAIL"

if [ ${#FAILED_PKGS[@]} -gt 0 ]; then
  echo ""
  echo "  Failed packages:"
  for pkg in "${FAILED_PKGS[@]}"; do
    echo "    - $pkg"
  done
  echo ""
  echo "Setup incomplete. Resolve the above before proceeding."
  exit 1
else
  echo ""
  echo "All packages installed successfully."
fi

 INSTALLING FROM requirements.txt
Using pip: pip 26.0.1 from /content/miniconda3/envs/dpcc/lib/python3.10/site-packages/pip (python 3.10)

Found requirements.txt -- 144 packages listed

  absl-py==2.1.0                          [ installed ]
  accelerate==0.31.0                      [ installed ]
  cachetools==5.3.3                       [ installed ]
  casadi==3.6.5                           [ installed ]
  certifi==2024.6.2                       [ installed ]
  charset-normalizer==3.3.2               [ installed ]
  chex==0.1.86                            [ installed ]
  clarabel==0.9.0                         [ installed ]
  click==8.1.7                            [ installed ]
  cloudpickle==3.0.0                      [ installed ]
  cmeel==0.53.3                           [ installed ]
  cmeel-assimp==5.3.1                     [ installed ]
  cmeel-boost==1.83.0                     [ installed ]
  cmeel-console-bridge==1.0.2.2           [ installed ]
  cmeel-octomap==1.9.8.2      

# Install D3IL

In [11]:
%%bash
D3IL_ROOT="/content/drive/MyDrive/DPCC/dpcc/d3il"
PIP="/content/miniconda3/envs/dpcc/bin/pip"

# Install d3il_sim core (correct subdirectory)
$PIP install -e "$D3IL_ROOT/environments/d3il" -q

# Install gym_avoiding env
$PIP install -e "$D3IL_ROOT/environments/d3il/envs/gym_avoiding_env" -q

echo "✓ D3IL installed"

✓ D3IL installed


# Set PYTHONPATH and MuJoCo renderer

In [12]:
import sys, os

CONDA_PYTHON = "/content/miniconda3/envs/dpcc/bin/python"
D3IL_ROOT    = "/content/drive/MyDrive/DPCC/dpcc/d3il"
GYM_AV_PATH  = f"{D3IL_ROOT}/environments/d3il/envs/gym_avoiding_env"

# Add paths needed for d3il absolute imports
sys.path.insert(0, D3IL_ROOT)
sys.path.insert(0, GYM_AV_PATH)

os.environ["MUJOCO_GL"]        = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"
os.environ["PYTHONPATH"]       = f"{D3IL_ROOT}:{GYM_AV_PATH}:" + os.environ.get("PYTHONPATH","")

print("✓ Paths and env vars set")

✓ Paths and env vars set


 # Full verification

In [13]:
%%bash
PYTHON="/content/miniconda3/envs/dpcc/bin/python"
D3IL_ROOT="/content/drive/MyDrive/DPCC/dpcc/d3il"
GYM_AV="/content/drive/MyDrive/DPCC/dpcc/d3il/environments/d3il/envs/gym_avoiding_env"

export PYTHONPATH="$D3IL_ROOT:$GYM_AV"
export MUJOCO_GL="egl"

$PYTHON - << 'EOF'
import warnings
warnings.filterwarnings("ignore")
import sys

print(f"Python:  {sys.version}")

import torch
print(f"✓ PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")

import mujoco
print(f"✓ mujoco {mujoco.__version__}")

from environments.d3il import d3il_sim
print("✓ environments.d3il.d3il_sim")

from environments.d3il.envs.gym_avoiding_env.gym_avoiding.envs.avoiding import ObstacleAvoidanceEnv
env = ObstacleAvoidanceEnv(render=False)
env.start()
obs = env.reset()
print(f"✓ ObstacleAvoidanceEnv OK | obs shape: {obs.shape}")

print("")
print("========================================")
print(" ALL CHECKS PASSED — ready to train!")
print("========================================")
EOF

Python:  3.10.20 (main, Mar 11 2026, 17:46:40) [GCC 14.3.0]
✓ PyTorch 2.6.0+cu124 | CUDA: True
✓ mujoco 2.3.7
✓ environments.d3il.d3il_sim
Final IK error (78 iterations):  7.2858622740627195e-06
Final IK error (0 iterations):  7.2858622740627195e-06
✓ ObstacleAvoidanceEnv OK | obs shape: (2,)

 ALL CHECKS PASSED — ready to train!



pybullet build time: Nov 28 2023 23:45:17


# Handle Gurobi

In [ ]:
# After getting license from <https://www.gurobi.com/academia/>
# Upload your gurobi.lic file to Drive, then:
import os
os.environ['GRB_LICENSE_FILE'] = '/content/drive/MyDrive/DPCC/gurobi.lic'

# Set Up wandb

In [21]:
import wandb
# Either login interactively:
#wandb.login()

# Or run fully offline (no account needed):
import os
os.environ['WANDB_MODE'] = 'offline'

# Dataset download

%%bash
ZIP="/content/drive/MyDrive/DPCC/dpcc/d3il/environments/dataset/data/dataset.zip"

echo "=== Top-level folders in zip ==="
unzip -l "$ZIP" | awk '{print $4}' | cut -d'/' -f1 | sort -u

echo ""
echo "=== File count per folder ==="
unzip -l "$ZIP" | awk '{print $4}' | cut -d'/' -f1 | sort | uniq -c | sort -rn

echo ""
echo "=== Total uncompressed size ==="
unzip -l "$ZIP" | tail -1

```
=== Top-level folders in zip ===

----
aligning
avoiding
inserting
Name
pushing
sorting
stacking

=== File count per folder ===
2638779 sorting
1377112 stacking
 392768 aligning
   2002 pushing
    802 inserting
     99 avoiding
      3
      1 Name
      1 ----

=== Total uncompressed size ===
17689115292                     4411562 files

Only Extract the avoiding

In [16]:
%%bash
DATASET_DIR="/content/drive/MyDrive/DPCC/dpcc/d3il/environments/dataset/data"
AVOIDING_DATA="$DATASET_DIR/avoiding/data"
ZIP_FILE="$DATASET_DIR/dataset.zip"

echo "========================================="
echo " D3IL DATASET SETUP"
echo " Source: ALRhub/d3il README.md"
echo " Link: https://drive.google.com/file/d/1SQhbhzV85zf_ltnQ8Cbge2lsSWInxVa8"
echo "========================================="
echo ""

# ── Step 1: Check if avoiding data already extracted ─────────────
if [ -d "$AVOIDING_DATA" ] && [ "$(ls -A $AVOIDING_DATA)" ]; then
    echo "✓ Dataset already extracted at: $AVOIDING_DATA"
    echo "  Files: $(ls $AVOIDING_DATA | wc -l) found"
    echo "  Skipping download and extraction."
    exit 0
fi

# ── Step 2: Check if zip already downloaded ───────────────────────
if [ -f "$ZIP_FILE" ]; then
    echo "✓ Zip already exists: $(du -sh $ZIP_FILE | cut -f1)"
    echo "  Skipping download."
else
    echo "Zip not found. Downloading..."
    mkdir -p "$DATASET_DIR"

    /content/miniconda3/envs/dpcc/bin/pip install gdown -q
    echo "✓ gdown ready"

    /content/miniconda3/envs/dpcc/bin/python -m gdown \
        "https://drive.google.com/uc?id=1SQhbhzV85zf_ltnQ8Cbge2lsSWInxVa8" \
        -O "$ZIP_FILE"

    if [ ! -f "$ZIP_FILE" ]; then
        echo "✗ Download failed — zip file not found"
        exit 1
    fi
    echo "✓ Downloaded: $(du -sh $ZIP_FILE | cut -f1)"
fi
echo ""

# ── Step 3: Extract ONLY avoiding/ to local Colab disk (fast) ────
echo "Extracting only avoiding/ to local disk first..."
LOCAL="/content/avoiding_tmp"
mkdir -p "$LOCAL"
unzip -q "$ZIP_FILE" "avoiding/*" -d "$LOCAL"

if [ $? -ne 0 ]; then
    echo "✗ Extraction failed"
    rm -rf "$LOCAL"
    exit 1
fi
echo "✓ Extracted locally: $(find $LOCAL -type f | wc -l) files | $(du -sh $LOCAL | cut -f1)"
echo ""

# ── Step 4: Copy only avoiding/ to Google Drive ───────────────────
echo "Copying avoiding/ to Google Drive..."
mkdir -p "$DATASET_DIR/avoiding"
cp -r "$LOCAL/avoiding/." "$DATASET_DIR/avoiding/"
echo "✓ Copied to Drive"

# ── Step 5: Cleanup local temp ────────────────────────────────────
rm -rf "$LOCAL"
echo "✓ Local temp cleaned up"
echo ""

# ── Step 6: Final verification ────────────────────────────────────
echo "========================================="
echo " VERIFICATION"
echo "========================================="
if [ -d "$AVOIDING_DATA" ] && [ "$(ls -A $AVOIDING_DATA)" ]; then
    echo "✓ avoiding/data found on Drive"
    echo "  Files: $(ls $AVOIDING_DATA | wc -l)"
    echo "  Sample: $(ls $AVOIDING_DATA | head -3)"
    echo ""
    echo "✓ Dataset ready — you can now run the smoke test."
else
    echo "✗ avoiding/data still not found."
    echo "  Actual structure:"
    find "$DATASET_DIR" -maxdepth 3 | sort | head -20
    exit 1
fi

 D3IL DATASET SETUP
 Source: ALRhub/d3il README.md
 Link: https://drive.google.com/file/d/1SQhbhzV85zf_ltnQ8Cbge2lsSWInxVa8

✓ Dataset already extracted at: /content/drive/MyDrive/DPCC/dpcc/d3il/environments/dataset/data/avoiding/data
  Files: 58 found
  Skipping download and extraction.


# Smoke Test Training Cell

In [17]:
%%bash
PYTHON="/content/miniconda3/envs/dpcc/bin/python"
PIP="/content/miniconda3/envs/dpcc/bin/pip"
DPCC="/content/drive/MyDrive/DPCC/dpcc"
D3IL_ROOT="$DPCC/d3il"
GYM_AV="$D3IL_ROOT/environments/d3il/envs/gym_avoiding_env"

export PYTHONPATH="$DPCC:$D3IL_ROOT:$GYM_AV"
export MUJOCO_GL="egl"
export WANDB_MODE="offline"
export MPLBACKEND="agg"

cd $DPCC

# Force reinstall gymnasium and minari to resolve potential import issues
$PIP install --force-reinstall gymnasium minari -q

$PYTHON - << 'EOF'
import matplotlib
matplotlib.use('agg')   # force non-interactive backend before any other import

import torch
import diffuser.utils as utils

exp = 'avoiding-d3il'
seeds = [5]

class Parser(utils.Parser):
    dataset: str = exp
    config: str = 'config.' + exp

for seed in seeds:
    args = Parser().parse_args(experiment='diffusion', seed=seed)
    torch.manual_seed(args.seed)

    # Smoke test overrides
    args.n_train_steps     = 10
    args.n_steps_per_epoch = 10
    args.batch_size        = 2

    print(f"Smoke test: seed={seed} | steps={args.n_train_steps} | batch={args.batch_size}")

    dataset_config = utils.Config(
        args.loader,
        savepath=(args.savepath, 'dataset_config.pkl'),
        env=args.dataset,
        horizon=args.horizon,
        normalizer=args.normalizer,
        preprocess_fns=args.preprocess_fns,
        use_padding=args.use_padding,
        max_path_length=args.max_path_length,
        include_returns=args.include_returns,
        returns_scale=args.max_path_length,
        discount=args.discount,
    )
    dataset = dataset_config()
    observation_dim = dataset.observation_dim
    action_dim      = dataset.action_dim
    goal_dim        = dataset.goal_dim

    model_config = utils.Config(
        args.model,
        savepath=(args.savepath, 'model_config.pkl'),
        horizon=args.horizon,
        transition_dim=observation_dim + action_dim,
        cond_dim=observation_dim,
        dim_mults=args.dim_mults,
        returns_condition=args.returns_condition,
        dim=args.dim,
        condition_dropout=args.condition_dropout,
        device=args.device,
    )

    diffusion_config = utils.Config(
        args.diffusion,
        savepath=(args.savepath, 'diffusion_config.pkl'),
        horizon=args.horizon,
        observation_dim=observation_dim,
        action_dim=action_dim,
        goal_dim=goal_dim,
        n_timesteps=args.n_diffusion_steps,
        loss_type=args.loss_type,
        clip_denoised=args.clip_denoised,
        predict_epsilon=args.predict_epsilon,
        action_weight=args.action_weight,
        loss_discount=args.loss_discount,
        returns_condition=args.returns_condition,
        condition_guidance_w=args.condition_guidance_w,
        device=args.device,
    )

    trainer_config = utils.Config(
        utils.Trainer,
        savepath=(args.savepath, 'trainer_config.pkl'),
        train_test_split=args.train_test_split,
        ema_decay=args.ema_decay,
        n_train_steps=args.n_train_steps,
        n_steps_per_epoch=args.n_steps_per_epoch,
        train_batch_size=args.batch_size,
        train_lr=args.learning_rate,
        gradient_accumulate_every=args.gradient_accumulate_every,
        results_folder=args.savepath,
    )

    model     = model_config()
    diffusion = diffusion_config(model)
    trainer   = trainer_config(diffusion, dataset)

    trainer.train()
    print(f"\n✓ Smoke test PASSED — seed {seed} completed {args.n_train_steps} steps")

print("\n========================================")
print(" SMOKE TEST COMPLETE — DPCC code works!")
print("========================================")
EOF

[ utils/setup ] Made savepath: logs/avoiding-d3il/diffusion/H8_K20_Dmodels.GaussianDiffusion/5
Smoke test: seed=5 | steps=10 | batch=2

[utils/config ] Config: <class 'diffuser.datasets.sequence.SequenceDataset'>
    discount: 0.99
    env: avoiding-d3il
    horizon: 8
    include_returns: True
    max_path_length: 150
    normalizer: LimitsNormalizer
    preprocess_fns: []
    returns_scale: 150
    use_padding: True

[ utils/config ] Saved config to: logs/avoiding-d3il/diffusion/H8_K20_Dmodels.GaussianDiffusion/5/dataset_config.pkl

[ datasets/buffer ] Fields:
    observations: (96, 150, 4)
    actions: (96, 150, 2)
    rewards: (96, 150, 1)
    terminals: (96, 150, 1)
    normed_observations: (96, 150, 4)
    normed_actions: (96, 150, 2)

[utils/config ] Config: <class 'diffuser.models.unet1d_temporal_cond.UNet1DTemporalCondModel'>
    cond_dim: 4
    condition_dropout: 0.25
    dim: 32
    dim_mults: (1, 2, 4, 8)
    horizon: 8
    returns_condition: False
    transition_dim: 6

[ 

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cmeel-boost 1.83.0 requires numpy~=1.26.0; python_version >= "3.9", but you have numpy 2.2.6 which is incompatible.
qpth 0.0.17 requires numpy<2,>=1, but you have numpy 2.2.6 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.1.3.1 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cuda-cupti-cu12 12.1.105 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-nvrtc-cu12==12.4.127; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cuda-nvrtc-cu12 12.1.105 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-runtime-cu12==12.4.127; pl

 # Train (using the conda env Python)

In [28]:
import torch
print(torch.cuda.is_available())       # Must be True
print(torch.cuda.get_device_name(0))   # Should show T4 or similar

True
Tesla T4


### Warning: use Real-time Streaming. Using !python instead of %%bash ensures that as soon as your script prints "Loading model..." or "Epoch 1", you see it immediately.

In [24]:
import os

# Set paths in the Python environment so they persist
os.environ['DPCC'] = "/content/drive/MyDrive/DPCC/dpcc"
os.environ['PYTHONPATH'] = f"{os.environ['DPCC']}:{os.environ['DPCC']}/d3il:{os.environ['DPCC']}/d3il/environments/d3il/envs/gym_avoiding_env"

# Set GPU and Backend flags
os.environ['MUJOCO_GL'] = "egl"
os.environ['PYOPENGL_PLATFORM'] = "egl"
os.environ['MPLBACKEND'] = "agg"
os.environ['WANDB_MODE'] = "offline"

# Change directory using python so it sticks for the next command
os.chdir(os.environ['DPCC'])

# Run the script using '!' to get real-time streaming output
!/content/miniconda3/envs/dpcc/bin/python scripts/train.py


[utils/config ] Config: <class 'diffuser.datasets.sequence.SequenceDataset'>
    discount: 0.99
    env: avoiding-d3il
    horizon: 8
    include_returns: True
    max_path_length: 150
    normalizer: LimitsNormalizer
    preprocess_fns: []
    returns_scale: 150
    use_padding: True

[ utils/config ] Saved config to: logs/avoiding-d3il/diffusion/H8_K20_Dmodels.GaussianDiffusion/5/dataset_config.pkl

[ datasets/buffer ] Fields:
    observations: (96, 150, 4)
    actions: (96, 150, 2)
    rewards: (96, 150, 1)
    terminals: (96, 150, 1)
    normed_observations: (96, 150, 4)
    normed_actions: (96, 150, 2)

[utils/config ] Config: <class 'diffuser.models.unet1d_temporal_cond.UNet1DTemporalCondModel'>
    cond_dim: 4
    condition_dropout: 0.25
    dim: 32
    dim_mults: (1, 2, 4, 8)
    horizon: 8
    returns_condition: False
    transition_dim: 6

[ utils/config ] Saved config to: logs/avoiding-d3il/diffusion/H8_K20_Dmodels.GaussianDiffusion/5/model_config.pkl


[utils/config ] Conf

# Set Up Virtual Display for MuJoCo Rendering

In [25]:
import os
import subprocess

# 1. Start Xvfb (Virtual Framebuffer)
# We check if it's already running to avoid multiple instances
try:
    subprocess.Popen(['Xvfb', ':1', '-screen', '0', '1400x900x24'])
except FileNotFoundError:
    !apt-get install -y xvfb
    subprocess.Popen(['Xvfb', ':1', '-screen', '0', '1400x900x24'])

# 2. Set Environment Variables
os.environ['DISPLAY'] = ':1'
os.environ['MUJOCO_GL'] = 'egl'
os.environ['PYOPENGL_PLATFORM'] = 'egl'

# 3. Verify MuJoCo using your CONDA python
# This ensures we use the version installed in your 'dpcc' environment
!/content/miniconda3/envs/dpcc/bin/python -c "import mujoco; print('MuJoCo version:', mujoco.__version__)"

MuJoCo version: 2.3.7


# Evaluate

In [31]:
# Downgrade NumPy to the latest 1.x version to support Pinocchio
!/content/miniconda3/envs/dpcc/bin/pip install "numpy<2"

  Using cached numpy-1.26.4-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.2 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.2.6
    Uninstalling numpy-2.2.6:
      Successfully uninstalled numpy-2.2.6


In [35]:
!ls -R /content/drive/MyDrive/DPCC/dpcc/logs/avoiding-d3il/diffusion/H8_K20_Dmodels.GaussianDiffusion/0/

ls: cannot access '/content/drive/MyDrive/DPCC/dpcc/logs/avoiding-d3il/diffusion/H8_K20_Dmodels.GaussianDiffusion/0/': No such file or directory


In [34]:
import os

# 1. Path Setup (Colab specific)
DPCC = "/content/drive/MyDrive/DPCC/dpcc"
D3IL_ROOT = f"{DPCC}/d3il"
GYM_AV = f"{D3IL_ROOT}/environments/d3il/envs/gym_avoiding_env"
PYTHON_BIN = "/content/miniconda3/envs/dpcc/bin/python"

# 2. Environment Setup (Matching your CPU logic but for GPU)
os.environ['MUJOCO_GL'] = 'egl'         # Colab GPU uses EGL
os.environ['PYOPENGL_PLATFORM'] = 'egl'
os.environ['WANDB_MODE'] = 'offline'
os.environ['PYTHONPATH'] = f"{DPCC}:{D3IL_ROOT}:{GYM_AV}"

# 3. Matplotlib Fix (The "Unset/Reset" logic from your CPU)
os.environ.pop('MPLBACKEND', None)
os.environ['MPLBACKEND'] = 'agg'

# 4. Execute with Streaming
os.chdir(DPCC)

print("--- Running Evaluation ---")
!{PYTHON_BIN} scripts/eval.py

print("\n--- Loading Results ---")
!{PYTHON_BIN} scripts/load_results.py

--- Running Evaluation ---
pybullet build time: Nov 28 2023 23:45:17

[ utils/serialization ] Loading model from logs/avoiding-d3il/diffusion/H8_K20_Dmodels.GaussianDiffusion/0

Traceback (most recent call last):
  File "/content/drive/MyDrive/DPCC/dpcc/scripts/eval.py", line 59, in <module>
    diffusion_experiment = utils.load_diffusion(args.loadbase, args.dataset, args.diffusion_loadpath, str(args.seed), epoch=args.diffusion_epoch, device=args.device)
  File "/content/drive/MyDrive/DPCC/dpcc/diffuser/utils/serialization.py", line 54, in load_diffusion
    dataset_config = load_config(*loadpath, 'dataset_config.pkl')
  File "/content/drive/MyDrive/DPCC/dpcc/diffuser/utils/serialization.py", line 36, in load_config
    config = pickle.load(open(loadpath, 'rb'))
FileNotFoundError: [Errno 2] No such file or directory: 'logs/avoiding-d3il/diffusion/H8_K20_Dmodels.GaussianDiffusion/0/dataset_config.pkl'

--- Loading Results ---
Traceback (most recent call last):
  File "/content/drive/MyD

#  Visualize

In [33]:
import os

# Define paths
dpcc_dir = "/content/drive/MyDrive/DPCC/dpcc"
d3il_dir = f"{dpcc_dir}/d3il"
gym_dir = f"{d3il_dir}/environments/d3il/envs/gym_avoiding_env"

# Setup environment variables
os.environ['MUJOCO_GL'] = "egl"
os.environ['PYOPENGL_PLATFORM'] = "egl"
os.environ['DISPLAY'] = ":1"
os.environ['MPLBACKEND'] = "agg"
os.environ['PYTHONPATH'] = f"{dpcc_dir}:{d3il_dir}:{gym_dir}"

# Move to the working directory
os.chdir(dpcc_dir)

# Stream the visualization output
!/content/miniconda3/envs/dpcc/bin/python scripts/visualize_data_constraints.py

Halfspace variant: top-right-hard
Number of feasible trajectories: 1/96, 1.04%
Halfspace variant: top-left-hard
Number of feasible trajectories: 0/96, 0.00%
Halfspace variant: both-hard
Number of feasible trajectories: 2/96, 2.08%
/content/drive/MyDrive/DPCC/dpcc/scripts/visualize_data_constraints.py:165: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [37]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import glob
import os

# Update this path to wherever your script saves the plots
plot_path = "/content/drive/MyDrive/DPCC/dpcc/logs/avoiding-d3il/plans/*/0/plots/*.png"
list_of_files = glob.glob(plot_path)

if list_of_files:
    # Sort by creation time to get the newest ones
    latest_files = sorted(list_of_files, key=os.path.getctime)[-3:]

    for img_path in latest_files:
        plt.figure(figsize=(10, 6))
        img = mpimg.imread(img_path)
        plt.imshow(img)
        plt.axis('off')
        plt.title(os.path.basename(img_path))
        plt.show()
else:
    print("No plot files found. Check your script's save directory.")

No plot files found. Check your script's save directory.
